#### Unsupervised

##### General
This Python function is a comprehensive evaluation tool designed for single-cell transcriptomics (specifically using the AnnData format). Its primary purpose is to identify which clustering algorithm and distance metric most accurately recapture known biological "ground truth" labels (like cell types) using the Adjusted Rand Index (ARI).

##### Data prep
Ground Truth Selection: It identifies the "true" labels in adata.obs[cell_type_col] and counts the unique clusters.
Representation: It extracts the data matrix. By default, it looks for PCA coordinates (X_pca), which is standard for single-cell data to reduce noise and improve performance for distance-based algorithms.

##### Methods
K-Means Variants (Global Centroids)
Standard K-Means: Uses Euclidean distance with k-means++ (smart seeding) vs. random seeding.
Spherical K-Means: By L2-normalizing the data and then running K-Means, it effectively clusters based on Cosine Similarity. This is often superior for high-dimensional gene expression where the "direction" of expression matters more than the magnitude.
Bisecting K-Means: A "top-down" approach that repeatedly splits the data. It is often more robust than standard K-Means when clusters are of unequal sizes.

Hierarchical Clustering (Connectivity & Linkage)
This section tests Agglomerative Clustering across a grid of distance metrics (Euclidean, Cosine, Manhattan, etc.) and linkage methods:
Ward: Minimizes variance within clusters (Euclidean only).
Complete/Average: Links clusters based on maximum or average distances between their members.

DBSCAN (Density-Based)
Unlike K-Means, DBSCAN does not need to know the number of clusters in advance. It identifies "dense" regions.
Grid Search: The function loops through various eps values (the maximum distance between two samples to be considered neighbors) across all selected metrics to find the density threshold that best matches the ground truth.

Leiden Clustering (Graph-Based)
The Leiden algorithm is the gold standard for single-cell analysis.It builds a Neighborhood Graph (connecting cells to their $k$ nearest neighbors).It performs a Resolution Sweep ($0.1$ to $3.0$). Higher resolution leads to more, smaller clusters; lower resolution leads to fewer, larger clusters.


##### Key Metrics & Evaluation Logic
The function uses the Adjusted Rand Index (ARI) as its primary benchmark:
ARI = Index - ExpectedIndex/MaxIndex - ExpectedIndex
Interpretation: An ARI of 1.0 indicates a perfect match to cell types; 0.0 indicates random labeling.
The Advantage: ARI is adjusted for chance, meaning it won't give a high score just because an algorithm created a lot of clusters.

##### Outputs
results_df (The Leaderboard): A sorted table showing which combination of Method + Metric + Parameter produced the highest ARI.
labels_df (The Predictions): A dataframe containing the actual cluster assignments for every cell across every tested method.
fig (The Visualization): An interactive Plotly scatter plot faceted by Method. This allows you to visually identify if a specific metric (like Cosine) consistently outperforms others across different algorithms

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from scintilla.clustering.benchmark import benchmark_clustering_methods


##### Example

In [ ]:
# 1. Check how many cells you have
print(f"Total cells: {adata_hvg.shape[0]}")

# 2. If you have > 10k cells, downsample for the benchmarking step
if adata_hvg.shape[0] > 10000:
    print("Dataset too large for Hierarchical Clustering benchmarking.")
    print("Subsampling to 10,000 cells...")
    adata_bench = sc.pp.subsample(adata_hvg, n_obs=10000, copy=True)
else:
    adata_bench = adata_hvg.copy()

# 3. Run the benchmark on the (sub)sampled dataset
df_stats, labels_dict, fig = benchmark_clustering_methods(
    adata_bench,
    cell_type_col='cell_type',
    use_rep='X_pca',
)

plt.show()


#### supervised

##### General
This Python code provides a high-level automated pipeline to determine which machine learning model and data representation (Gene Space vs. PCA Space) best classify cell types in a single-cell dataset.

##### functions
calculate_advanced_metrics function 
This is a helper function that looks at specific types of errors

benchmark_models_comprehensive
This is the "main" function. It treats your AnnData object as a laboratory, testing multiple models under different conditions.
The function compares two ways of "looking" at the data: Gene Space vs PCA Space
The Benchmarking Loop: 1. Scaling 2. Stratified 3. Model Arena
Model Arena:
Logistic Regression (LogReg): Good baseline.
RandomForest: Excellent for non-linear gene relationships.
SVM: Finds complex boundaries between cell types.
MLP (Deep Learning): A small neural network for finding deep patterns.
LDA/QDA: Statistical models that assume specific data distributions.

##### Metrics & Visualization
For every model/space combination, it calculates Accuracy, Precision, Recall, F1-Score, FDR, and FNR.

In [ ]:
from IPython.display import display
from scintilla.classification.benchmark import benchmark_models_comprehensive


##### Example

In [ ]:
print(f"Total cells: {adata_hvg.shape[0]}")

# If we have > 10k cells, downsample for the benchmarking step
if adata_hvg.shape[0] > 10000:
    print("Dataset too large for benchmarking. Subsampling to 10,000 cells...")
    adata_bench = sc.pp.subsample(adata_hvg, n_obs=10000, copy=True)
else:
    adata_bench = adata_hvg.copy()

df_results, fig = benchmark_models_comprehensive(
    adata_bench,
    target_col="cell_type",
)

display(df_results.round(3))
plt.show()
